# Accessing Data

In [17]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [18]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

## Holiday Function

In [19]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

In [20]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Relative Ranking
- for every hour (24 hour period) rank each data point relative to the others on the amount of demand used
- produces a list of 1 - 122, this ranking is done for each hour
- relative ts = [56, 74, 2, ...]
- day 56 out of the list (1-122) had the highest electricity demand at 12pm, day 74 had the highest demand at 12pm etc etc
- christmas = day 30
- at 12pm, day 30 ranked 12 : relative rank = n/122 (12/122 = 0.1) therefore at 12pm christmas day electricity was in the lowest 10% (at 12pm on christmas day electricity demand was lower relative another day in those 122 days)

## Downloading libraries

In [21]:
import os
import pandas as pdf
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from dateutil.easter import easter

## Year intervals

In [22]:
intervals = [[y, y+1] for y in range(2004, 2018)]

## Function: Public Holiday Relative Ranking Plot

In [23]:
def holiday_relative_rank_plot(demand, info, station, years, holiday_func, holiday_name):
    """
    Computes the hour-stratified relative ranking of electricity demand
    for a specific public holiday across multiple years.
    Produces a 24-hour curve of relative rank (0 = lowest, 1 = highest).
    """

    # Ensure datetime index
    demand.index = pd.to_datetime(demand.index)

    # Hourly mean demand
    hourly = demand[[station]].resample("h").mean()

    # 30-day window around holiday for each year
    windows = []
    for year in years:
        ref_date = holiday_func(year)
        start = ref_date - pd.Timedelta(days=30)
        end   = ref_date + pd.Timedelta(days=30)
        win = hourly.loc[start:end].copy()
        windows.append(win)

    combined = pd.concat(windows)
    combined["date"] = combined.index.date
    combined["hour"] = combined.index.hour

    # --- Compute relative rank for EACH hour ---
    combined["rank"] = combined.groupby("hour")[station].rank(method="average")

    # Convert to percentile
    n_days = combined.groupby("hour")["date"].transform("nunique")
    combined["relative_rank"] = combined["rank"] / n_days

    # --- Extract holiday profile (mean across years) ---
    holiday_hours = []
    for year in years:
        ref_date = holiday_func(year)
        expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")

        daily = combined["relative_rank"].reindex(expected_hours)
        daily.index = range(24)
        holiday_hours.append(daily)

    holiday_profile = pd.concat(holiday_hours, axis=1).mean(axis=1)

    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(10,4))

    ax.plot(
        holiday_profile.index,
        holiday_profile.values,
        color="red",
        marker="o",
        linewidth=2,
        label=f"{holiday_name}"
    )

    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_yticklabels(["Lowest", "25%", "50%", "75%", "Highest"])

    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(f"{full_name} Relative Demand Rank: {holiday_name} ({years[0]}–{years[-1]})")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Relative Rank (0 = lowest, 1 = highest)")
    ax.legend()

    return fig

In [25]:
station = "BLAKE"
years = [2006, 2007]
holiday_name = "Christmas Day"
holiday_func = HOLIDAYS_VIC[holiday_name]

fig = holiday_relative_rank_plot(demand, info, station, years, holiday_func, holiday_name)
plt.close()